# Case study: first SQL questions about the Berlin Airbnb listings

*Session 3, block 1 · Author: course team · Licence: CC BY 4.0*

**Why.** A housing analyst of the city wants a first picture of short-term rentals in Berlin. Before any statistics,
she needs to know what the three tables contain, whether their keys hold, and how to pick out the rows that a
question is about. This notebook does these first steps in SQL on the course database, with complete solutions.

The data (Inside Airbnb, Berlin snapshot of 26 June 2026, CC BY 4.0) are in `case-study/data/airbnb.sqlite`, built
by [`make_airbnb_sqlite.py`](make_airbnb_sqlite.py) from the files of `case-study/prepare_airbnb.py`:

- `listings`: 12,776 listings with district, room type, guests, price per night, minimum stay, reviews and
  registration status;
- `calendar`: one row per listing and night for the next 365 days: is the night still available (1) or not (0)?
- `reviews`: number of reviews per listing and month since 2009.

**Steps**

1. Look at the tables and check their keys.
2. First queries: filters on numbers, text and dates.
3. Missing values: which listings show no price?

In [1]:
import sqlite3
import subprocess
import sys
from pathlib import Path

import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DB = ROOT / "case-study" / "data" / "airbnb.sqlite"
if not DB.exists():   # build the course database once (needs case-study/prepare_airbnb.py to have run)
    subprocess.run([sys.executable, ROOT / "sessions/03-sql/workbooks/make_airbnb_sqlite.py"], check=True)

con = sqlite3.connect(DB)


def q(sql, params=()):
    """Run a query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, con, params=params)


q("SELECT (SELECT COUNT(*) FROM listings) AS listings, (SELECT COUNT(*) FROM calendar) AS calendar_rows, "
  "(SELECT COUNT(*) FROM reviews) AS review_months")

,listings,calendar_rows,review_months
0,12776,4692075,226557


## 1. Look at the tables and check their keys

`PRAGMA table_info(table)` lists the columns and their declared types. Remember that SQLite stores dates as text and
true/false as 1/0.

In [2]:
q("PRAGMA table_info(calendar)")[["name", "type"]]

,name,type
0,listing_id,INTEGER
1,date,TEXT
2,available,INTEGER


In [3]:
q("SELECT * FROM reviews WHERE listing_id = 3176 ORDER BY month LIMIT 5")

,listing_id,month,n_reviews
0,3176,2009-06-01,1
1,3176,2010-11-01,2
2,3176,2010-12-01,1
3,3176,2011-01-01,3
4,3176,2011-02-01,1


A **primary key** must be unique. `COUNT(DISTINCT column)` counts the different values; if it equals the number of
rows, no value occurs twice. For the composite key of the calendar, the subquery in parentheses first keeps the
distinct pairs, and the outer query counts them.

In [4]:
q("""
SELECT (SELECT COUNT(*) FROM listings)                                   AS n_listings,
       (SELECT COUNT(DISTINCT id) FROM listings)                         AS n_ids,
       (SELECT COUNT(*) FROM calendar)                                   AS n_calendar_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, date FROM calendar)) AS n_calendar_keys,
       (SELECT COUNT(*) FROM reviews)                                    AS n_review_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, month FROM reviews)) AS n_review_keys
""")

,n_listings,n_ids,n_calendar_rows,n_calendar_keys,n_review_rows,n_review_keys
0,12776,12776,4692075,4692075,226557,226557


All three keys hold. Text dates sort correctly, so `MIN` and `MAX` work on them:

In [5]:
q("SELECT MIN(date) AS first_night, MAX(date) AS last_night, COUNT(DISTINCT date) AS n_nights FROM calendar")

,first_night,last_night,n_nights
0,2026-06-26,2027-07-02,372


The calendar has 372 different dates, although each listing has exactly 365 rows: Inside Airbnb scraped the
listings on different days between 26 June and 3 July 2026, and each calendar starts on the day its listing was
scraped. Questions about "the next weeks" therefore mean slightly different weeks for different listings.

## 2. First queries: filters on numbers, text and dates

A family of four looks for an entire flat in Friedrichshain-Kreuzberg for a weekend: a minimum stay of at most two
nights, at least four guests, at most €150 a night, and at least 20 reviews. We show the id and the numbers, not
the title: hosts sometimes write their own name into the title, and the analysis does not need it.

In [6]:
q("""
SELECT id, neighbourhood, accommodates, price, number_of_reviews, review_scores_rating
FROM listings
WHERE district = 'Friedrichshain-Kreuzberg'
  AND room_type = 'Entire home/apt'
  AND minimum_nights <= 2 AND accommodates >= 4 AND price <= 150 AND number_of_reviews >= 20
ORDER BY review_scores_rating DESC, number_of_reviews DESC
LIMIT 5
""")

,id,neighbourhood,accommodates,price,number_of_reviews,review_scores_rating
0,1350839618181161054,Frankfurter Allee Süd FK,8,135.18,90,4.94
1,551962219978207195,Frankfurter Allee Süd FK,4,83.70,32,4.94
2,20660494,Tempelhofer Vorstadt,4,128.67,361,4.88
3,51830435,Frankfurter Allee Süd FK,6,128.00,271,4.86
4,48732044,Karl-Marx-Allee-Süd,6,129.00,298,4.85


Is the best-rated flat still free for the first weekend of August 2026? Dates are text in the form `'YYYY-MM-DD'`,
so `BETWEEN` on text works like a date range. `strftime('%w', date)` returns the weekday (0 is Sunday).

In [7]:
best = int(q("""
SELECT id FROM listings
WHERE district = 'Friedrichshain-Kreuzberg' AND room_type = 'Entire home/apt'
  AND minimum_nights <= 2 AND accommodates >= 4 AND price <= 150 AND number_of_reviews >= 20
ORDER BY review_scores_rating DESC, number_of_reviews DESC
LIMIT 1
""").iloc[0, 0])

q("""
SELECT date, strftime('%w', date) AS weekday, available
FROM calendar
WHERE listing_id = ? AND date BETWEEN '2026-07-31' AND '2026-08-02'
ORDER BY date
""", params=(best,))

,date,weekday,available
0,2026-07-31,5,0
1,2026-08-01,6,0
2,2026-08-02,0,0


The `?` is a placeholder for a value that Python passes separately (block 2 explains why this is safer than
pasting the id into the text). Text patterns with `LIKE`: how many listing titles mention a garden or a terrace,
in English or German? `%` stands for any sequence of characters, and `LIKE` ignores upper and lower case.

In [8]:
q("""
SELECT COUNT(*) AS n
FROM listings
WHERE name LIKE '%garden%' OR name LIKE '%garten%' OR name LIKE '%terrace%' OR name LIKE '%terrasse%'
""")

,n
0,640


## 3. Missing values: which listings show no price?

About a third of the listings have no price. Is a missing price random, or does it mean something? A listing
whose calendar has not a single available night in the coming year (`availability_365 = 0`) cannot show the price
of a next free night.

In [9]:
q("""
SELECT (SELECT COUNT(*) FROM listings WHERE price IS NULL)                              AS no_price,
       (SELECT COUNT(*) FROM listings WHERE price IS NULL AND availability_365 = 0)     AS no_price_no_free_night,
       (SELECT COUNT(*) FROM listings WHERE price IS NOT NULL AND availability_365 = 0) AS price_no_free_night
""")

,no_price,no_price_no_free_night,price_no_free_night
0,4335,3488,2


Four in five listings without a price have no free night at all: they are probably not offered at the moment. Any
statement about "the price of a listing in Berlin" therefore describes the listings that can be booked, not all
listings. Session 4 returns to such missing values.

`price = NULL` would find nothing: a comparison with NULL is unknown, never true.

In [10]:
q("SELECT COUNT(*) AS wrong FROM listings WHERE price = NULL")

,wrong
0,0


**Summary.** The keys of the three tables hold; text dates can be filtered and sorted like dates; and the missing
prices are concentrated in listings that cannot be booked. Block 2 combines the tables with joins and aggregates
them.